# Supply Chain Shipping Delay Risk — Logistic Regression

This notebook applies **Logistic Regression** to the same supply-chain project.

The earlier regression notebook predicted the exact number of delay days:

`shipping_delay_days`

For Logistic Regression, we convert the problem into a **binary classification problem**:

- `0` = Lower Delay Risk
- `1` = High Delay Risk

For this notebook:

**High Delay = shipping delay >= 7 days**

The model answers:

> **Will this future trade-route operation experience a high shipping delay (7 days or more)?**

Topics covered:

- Binary classification target creation
- Train/test split
- Preprocessing
- Logistic Regression
- Sigmoid probability
- Confusion Matrix
- Accuracy
- Precision
- Recall / Sensitivity
- Specificity
- F1 Score
- ROC Curve
- AUC
- Threshold analysis
- Hyperparameter tuning (`C`)
- Feature coefficient interpretation
- Optional multiclass Logistic Regression


## 1. Imports


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 2. Load the processed dataset

This notebook starts from the same cleaned and feature-engineered dataset used by the Linear Regression notebook.


In [ ]:
def find_processed_file():
    candidates = [
        Path.cwd() / "data" / "processed" / "supply_chain_ml_ready.csv",
        Path.cwd().parent / "data" / "processed" / "supply_chain_ml_ready.csv",
        Path("/mnt/data/data/processed/supply_chain_ml_ready.csv"),
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "Could not find supply_chain_ml_ready.csv. Run the EDA notebook first."
    )

DATA_PATH = find_processed_file()
print(f"Using dataset: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Remove redundant / leakage columns

We do not use columns that directly contain or reveal the final delay outcome.

Important:

- `shipping_delay_days` is used only to create the classification target.
- `delay_category` must not be used as an input because it is derived from shipping delay.
- `container_shortage_score` is redundant with `container_availability_index`.
- `route_status` may contain post-outcome information.
- `operational_risk_score` is excluded because it combines individual risk variables already used by the model.


In [ ]:
REGRESSION_TARGET = "shipping_delay_days"

df.drop(
    columns=["container_shortage_score"],
    inplace=True,
    errors="ignore"
)

print("shipping_delay_days present:", REGRESSION_TARGET in df.columns)
print("container_shortage_score present:", "container_shortage_score" in df.columns)


## 4. Create the Binary Classification Target

Logistic Regression needs a categorical target.

We define:

- `0` → shipping delay < 7 days
- `1` → shipping delay >= 7 days

The new target is:

`high_delay_flag`

This target is created only for classification.  
The original `shipping_delay_days` is **not** passed to the model as an input feature.


In [ ]:
DELAY_THRESHOLD_DAYS = 7

df["high_delay_flag"] = (
    df[REGRESSION_TARGET] >= DELAY_THRESHOLD_DAYS
).astype(int)

CLASS_TARGET = "high_delay_flag"

print(df[CLASS_TARGET].value_counts())
print()
print("Class percentages:")
print(df[CLASS_TARGET].value_counts(normalize=True).mul(100).round(2))


In [ ]:
class_summary = (
    df.groupby(CLASS_TARGET)[REGRESSION_TARGET]
      .agg(["count", "mean", "median", "min", "max"])
)

display(class_summary)


### Business meaning

The model no longer predicts the exact number of days.

It now predicts:

> **Probability that a route operation will have a shipping delay of at least 7 days.**


## 5. Select Features

We keep the same interpretable operational and environmental predictors used in the Linear Regression notebook.


In [ ]:
numeric_features = [
    "trade_volume_tonnes",
    "freight_cost_per_tonne",
    "container_availability_index",
    "port_congestion_index",
    "fuel_cost_index",
    "commodity_stress_index",
    "weather_disruption_score",
    "geopolitical_risk_score",
    "distance_km",
    "estimated_transit_days",
    "month",
]

categorical_features = [
    "shipping_method",
    "trade_route_type",
]

model_features = numeric_features + categorical_features

X = df[model_features].copy()
y = df[CLASS_TARGET].copy()

print("Input features:", len(model_features))
print("Classification target:", CLASS_TARGET)
display(X.head())


## 6. Time-Aware Train/Test Split

Because the business goal is to predict future route operations, we keep the same chronological split:

- First 80% → Training
- Last 20% → Testing

This avoids mixing future records into training.


In [ ]:
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

train_dates = df["date"].iloc[:split_index]
test_dates = df["date"].iloc[split_index:]

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows : {len(X_test):,}")
print(f"Train period : {train_dates.min().date()} to {train_dates.max().date()}")
print(f"Test period  : {test_dates.min().date()} to {test_dates.max().date()}")

print()
print("Training class distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print()
print("Testing class distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))


## 7. Preprocessing

Logistic Regression requires numerical input.

- Numeric columns → StandardScaler
- Categorical columns → OneHotEncoder
- `drop="first"` prevents unnecessary dummy-variable redundancy.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_features,
        ),
    ],
    verbose_feature_names_out=False,
)


## 8. Baseline Classifier

Before training Logistic Regression, create a simple baseline that always predicts the most common training class.

A useful model should outperform this baseline.


In [ ]:
baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

print("Baseline Accuracy:", round(accuracy_score(y_test, baseline_pred), 4))


# 9. Logistic Regression

Logistic Regression first creates a linear score:

`z = b0 + b1x1 + b2x2 + ...`

Then the **Sigmoid function** converts that score to a probability between 0 and 1:

`p = 1 / (1 + exp(-z))`

The default classification threshold is usually **0.50**.


In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=5000,
                solver="liblinear",
                random_state=42,
            ),
        ),
    ]
)

logistic_model.fit(X_train, y_train)

# Predicted class: 0 or 1
y_pred = logistic_model.predict(X_test)

# Probability of class 1 = High Delay
y_prob = logistic_model.predict_proba(X_test)[:, 1]

print("First 10 predicted probabilities:")
print(np.round(y_prob[:10], 4))

print()
print("First 10 predicted classes:")
print(y_pred[:10])


## 10. Evaluation Metrics

For classification, we use:

- Accuracy
- Precision
- Recall / Sensitivity
- Specificity
- F1 Score
- ROC-AUC


In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
specificity = tn / (tn + fp)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_prob)

metrics_table = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall / Sensitivity",
        "Specificity",
        "F1 Score",
        "ROC-AUC",
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        specificity,
        f1,
        auc,
    ],
})

display(metrics_table)


### How to interpret the metrics

- **Accuracy** → Out of all route operations, how many were classified correctly?
- **Precision** → When the model says "High Delay", how often is it correct?
- **Recall** → Out of all actual High Delay operations, how many did the model detect?
- **Specificity** → Out of all actual Lower Delay operations, how many did the model correctly identify?
- **F1** → Balance between Precision and Recall.
- **ROC-AUC** → How well the model separates High Delay from Lower Delay across thresholds.


## 11. Confusion Matrix

Confusion Matrix:

- TP = Correctly predicted High Delay
- TN = Correctly predicted Lower Delay
- FP = Predicted High Delay, but actually Lower Delay
- FN = Predicted Lower Delay, but actually High Delay


In [ ]:
cm = confusion_matrix(y_test, y_pred)

confusion_df = pd.DataFrame(
    cm,
    index=["Actual Lower Delay (0)", "Actual High Delay (1)"],
    columns=["Predicted Lower Delay (0)", "Predicted High Delay (1)"],
)

display(confusion_df)

print(f"True Negative : {tn}")
print(f"False Positive: {fp}")
print(f"False Negative: {fn}")
print(f"True Positive : {tp}")


In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(cm)

plt.xticks(
    [0, 1],
    ["Predicted 0", "Predicted 1"],
)
plt.yticks(
    [0, 1],
    ["Actual 0", "Actual 1"],
)

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
            fontsize=14,
        )

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()


## 12. Classification Report


In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Lower Delay", "High Delay"],
        zero_division=0,
    )
)


# 13. ROC Curve and AUC

ROC plots:

- X-axis → False Positive Rate
- Y-axis → True Positive Rate / Recall

AUC summarizes the ROC curve.

- AUC = 1.0 → Perfect separation
- AUC = 0.5 → Similar to random ranking
- Higher AUC → Better class separation


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"Logistic Regression AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — High Shipping Delay Classification")
plt.legend()
plt.tight_layout()
plt.show()

print("ROC-AUC:", round(roc_auc, 4))


# 14. Probability Threshold Analysis

The default threshold is 0.50.

Changing the threshold changes Precision and Recall.

Lower threshold:
- predicts more High Delay cases
- usually increases Recall
- may increase False Positives

Higher threshold:
- predicts fewer High Delay cases
- may increase Precision
- may increase False Negatives

This section is educational.  
Do not choose the final production threshold only by looking at the test set.


In [ ]:
threshold_results = []

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    threshold_pred = (y_prob >= threshold).astype(int)

    tn_t, fp_t, fn_t, tp_t = confusion_matrix(
        y_test,
        threshold_pred,
        labels=[0, 1],
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, threshold_pred),
        "Precision": precision_score(
            y_test,
            threshold_pred,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_test,
            threshold_pred,
            zero_division=0,
        ),
        "Specificity": tn_t / (tn_t + fp_t),
        "F1": f1_score(
            y_test,
            threshold_pred,
            zero_division=0,
        ),
        "False_Positive": fp_t,
        "False_Negative": fn_t,
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    threshold_df["Threshold"],
    threshold_df["Precision"],
    marker="o",
    label="Precision",
)

plt.plot(
    threshold_df["Threshold"],
    threshold_df["Recall"],
    marker="o",
    label="Recall",
)

plt.plot(
    threshold_df["Threshold"],
    threshold_df["F1"],
    marker="o",
    label="F1",
)

plt.xlabel("Probability Threshold")
plt.ylabel("Metric Value")
plt.title("Threshold Effect on Classification Metrics")
plt.legend()
plt.tight_layout()
plt.show()


# 15. Hyperparameter Tuning

For Logistic Regression, an important hyperparameter is **C**.

`C` is the inverse of regularization strength.

- Small C → stronger regularization
- Large C → weaker regularization

We tune C using TimeSeriesSplit on the training period only.

This avoids using the test period to choose model settings.


In [ ]:
time_cv = TimeSeriesSplit(n_splits=5)

tuned_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=5000,
                solver="liblinear",
                random_state=42,
            ),
        ),
    ]
)

param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__penalty": ["l1", "l2"],
}

logistic_search = GridSearchCV(
    tuned_pipeline,
    param_grid=param_grid,
    cv=time_cv,
    scoring="roc_auc",
    n_jobs=-1,
)

logistic_search.fit(X_train, y_train)

print("Best parameters:", logistic_search.best_params_)
print("Best CV ROC-AUC:", round(logistic_search.best_score_, 4))


## 16. Evaluate the Tuned Logistic Regression Model


In [ ]:
best_logistic_model = logistic_search.best_estimator_

tuned_pred = best_logistic_model.predict(X_test)
tuned_prob = best_logistic_model.predict_proba(X_test)[:, 1]

tn2, fp2, fn2, tp2 = confusion_matrix(
    y_test,
    tuned_pred,
    labels=[0, 1],
).ravel()

tuned_metrics = {
    "Accuracy": accuracy_score(y_test, tuned_pred),
    "Precision": precision_score(y_test, tuned_pred, zero_division=0),
    "Recall": recall_score(y_test, tuned_pred, zero_division=0),
    "Specificity": tn2 / (tn2 + fp2),
    "F1": f1_score(y_test, tuned_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, tuned_prob),
}

display(
    pd.DataFrame(
        tuned_metrics.items(),
        columns=["Metric", "Tuned_Model_Value"],
    )
)


## 17. Compare Default vs Tuned Logistic Regression


In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1",
        "ROC_AUC",
    ],
    "Default_Logistic": [
        accuracy,
        precision,
        recall,
        specificity,
        f1,
        auc,
    ],
    "Tuned_Logistic": [
        tuned_metrics["Accuracy"],
        tuned_metrics["Precision"],
        tuned_metrics["Recall"],
        tuned_metrics["Specificity"],
        tuned_metrics["F1"],
        tuned_metrics["ROC_AUC"],
    ],
})

display(comparison)


# 18. Feature Coefficients

Logistic Regression coefficients tell us how each feature changes the **log-odds** of High Delay.

Simple interpretation:

- Positive coefficient → tends to increase probability of High Delay
- Negative coefficient → tends to decrease probability of High Delay
- Larger absolute coefficient → stronger linear influence, after preprocessing

Because numeric features are standardized, their coefficient magnitudes are easier to compare.


In [ ]:
feature_names = (
    best_logistic_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    best_logistic_model
    .named_steps["model"]
    .coef_[0]
)

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})

coef_df["absolute_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    "absolute_coefficient",
    ascending=False,
)

display(coef_df.head(20))


In [ ]:
top_coef = coef_df.head(15).sort_values("coefficient")

plt.figure(figsize=(9, 7))
plt.barh(
    top_coef["feature"],
    top_coef["coefficient"],
)

plt.axvline(0, linewidth=1)
plt.xlabel("Logistic Regression Coefficient")
plt.ylabel("Feature")
plt.title("Most Influential Logistic Regression Features")
plt.tight_layout()
plt.show()


# 19. Business Interpretation

This Logistic Regression model predicts the **risk of a high shipping delay**, not the exact delay duration.

Example:

If the model returns:

`High Delay Probability = 0.82`

we can explain:

> Based on the operational and environmental conditions of this trade-route operation, the model estimates an 82% probability that the shipping delay will be 7 days or more.

This can help logistics teams:

- prioritize risky shipments
- alert customers earlier
- investigate congested routes
- prepare alternate transportation plans
- allocate containers and operational resources


# 20. Binary Regression vs Binary Classification in This Project

### Linear Regression

Target:

`shipping_delay_days`

Example prediction:

`8.4 days`

Question:

> How many days will the shipment be delayed?

### Logistic Regression

Target:

`high_delay_flag`

Example prediction:

`82% probability of High Delay`

Question:

> Is this shipment likely to have a high delay?

Both approaches are valid, but they answer different business questions.


# 21. Optional: Multiclass Logistic Regression

If you want to classify shipments into several delay-risk classes instead of only two classes, create:

- 0 = Low
- 1 = Moderate
- 2 = High
- 3 = Severe

For multiclass Logistic Regression, scikit-learn handles the class probabilities internally.

This section is optional and separate from the main binary model.


In [ ]:
# OPTIONAL MULTICLASS TARGET

df["delay_class"] = pd.cut(
    df[REGRESSION_TARGET],
    bins=[-np.inf, 2, 5, 10, np.inf],
    labels=[0, 1, 2, 3],
).astype(int)

multi_y = df["delay_class"]

multi_y_train = multi_y.iloc[:split_index]
multi_y_test = multi_y.iloc[split_index:]

multiclass_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=5000,
                solver="lbfgs",
            ),
        ),
    ]
)

multiclass_model.fit(X_train, multi_y_train)

multi_pred = multiclass_model.predict(X_test)

print(
    classification_report(
        multi_y_test,
        multi_pred,
        target_names=[
            "Low",
            "Moderate",
            "High",
            "Severe",
        ],
        zero_division=0,
    )
)


# 22. Final Summary

The Logistic Regression workflow for this supply-chain project is:

1. Start with cleaned EDA data
2. Convert shipping delay into a binary business target
3. Keep only pre-outcome features
4. Split data chronologically
5. Scale numeric columns
6. Encode categorical columns
7. Train Logistic Regression
8. Predict classes and probabilities
9. Evaluate using Confusion Matrix
10. Calculate Accuracy, Precision, Recall, Specificity and F1
11. Plot ROC curve
12. Calculate AUC
13. Study probability thresholds
14. Tune regularization using C
15. Interpret feature coefficients

The main classification target is:

> **Will a trade-route operation experience a shipping delay of 7 days or more?**
